In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Linear Models

## Data Loading and Preprocessing

In [18]:
import sys
sys.path.append("..")
from src.data import load_data, compute_returns, create_sequences
import numpy as np

In [19]:
df = load_data()
df_returns = df.copy()
df_returns = compute_returns(df_returns)
df_returns = df_returns.dropna()

# Scale asset values between -1 and 1
from sklearn.preprocessing import StandardScaler
stock_cols = ['AMZN', 'DPZ', 'BTC', 'NFLX']
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_returns[stock_cols])

# Split time series data into training samples
X, y = create_sequences(scaled_data, window_size=60)

# Prep data for linear models
n_samples, window, n_features = X.shape

X_lin = X.reshape(n_samples, window * n_features)

# Train/val split WITHOUT shuffling (as we're dealing with sequential time series data)
split = int(0.8 * len(X))
X_train_lin, X_val_lin = X_lin[:split], X_lin[split:]
y_train_lin, y_val_lin = y[:split], y[split:]

In [20]:
def mse_per_asset(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2, axis=0)

## Ridge regression

In [21]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

model = Ridge(alpha=0.1)
model.fit(X_train_lin, y_train_lin)

y_pred = model.predict(X_val_lin)
mse_each = mse_per_asset(y_val_lin, y_pred)
print("Ridge MSE per asset:", mse_each)
print("Ridge overall MSE:", np.mean(mse_each)) 

Ridge MSE per asset: [1.61797019 1.66726092 0.56338731 1.33399332]
Ridge overall MSE: 1.2956529344530843


## Lasso

In [22]:
from sklearn.linear_model import Lasso

stock_cols = ['AMZN','DPZ','BTC','NFLX']
mse_lasso = {}

for j, name in enumerate(stock_cols):
    model = Lasso(alpha=0.001)
    # train just that column
    model.fit(X_train_lin, y_train_lin[:, j])

    y_pred = model.predict(X_val_lin)
    mse_lasso[name] = mean_squared_error(y_val_lin[:, j], y_pred)

print("Lasso MSE per asset:", mse_lasso)

Lasso MSE per asset: {'AMZN': 1.6016358062701914, 'DPZ': 1.646832265602723, 'BTC': 0.5367243754515502, 'NFLX': 1.3141417741489596}


## ElasticNet

In [23]:
from sklearn.linear_model import ElasticNet

mse_elastic = {}

for j, name in enumerate(stock_cols):
    model = ElasticNet(alpha=0.001, l1_ratio=0.5)
    model.fit(X_train_lin, y_train_lin[:, j])

    y_pred = model.predict(X_val_lin)
    mse_elastic[name] = mean_squared_error(y_val_lin[:, j], y_pred)

print("ElasticNet MSE per asset:", mse_elastic)

ElasticNet MSE per asset: {'AMZN': 1.6092566914550004, 'DPZ': 1.656224012670999, 'BTC': 0.5493011449386841, 'NFLX': 1.3233659555242068}


## AR

In [24]:
from statsmodels.tsa.ar_model import AutoReg

mse_ar = {}
for j, name in enumerate(stock_cols):
    series = scaled_data[:, j]             # scaled returns for this asset

    # compute train/validation split indices as before
    start_idx = window + split
    end_idx   = start_idx + len(y_val_lin[:, 0]) - 1

    ar_train = series[:start_idx]
    model = AutoReg(ar_train, lags=30)
    fit   = model.fit()

    pred  = fit.predict(start=start_idx, end=end_idx)
    mse_ar[name] = mean_squared_error(y_val_lin[:, j], pred)

print("AR MSE per asset:", mse_ar)

AR MSE per asset: {'AMZN': 1.3488077065272026, 'DPZ': 1.3365258304534091, 'BTC': 0.1668287775921179, 'NFLX': 1.0657749949474773}
